# Corpus Summary and Utterance Durations — Tables 5 and 9

Generates the two corpus-level tables of the thesis directly from the DVSS corpus,
using the pipeline's own loader and tokeniser (`data/extract.py`, `data/transform.py`)
so the counts are by construction the ones the models were trained on.

- **Table 5** — full C1 corpus: utterances per mode and vocabulary size |V| per partition.
- **Table 9** — utterance duration statistics, computed from the EMG signal length
  (8 channels @ 1000 Hz), in seconds.

Requires the DVSS corpus (see `data/README.md`). No GPU, no API.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "extract.py").exists() and (cand / "train.py").exists():
            return cand
    raise RuntimeError(f"Could not locate repo root from {p}")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "data"))

from extract import load_dataset          # noqa: E402
from transform import transform_dataset   # noqa: E402

SAMPLE_RATE = 1000   # Hz — features.sample_rate in configs/base.yaml
print("repo root:", REPO_ROOT)


## Load the five partitions of C1


In [ ]:
def load(vocab_type, mode, group=None):
    return load_dataset(
        "DVSS", vocabulary_type=vocab_type, mode=mode, group=group,
        require_text=True, require_emg=True,
    )


partitions = {
    ("C1_closed",    "Silent"):    load("closed_vocab", "silent"),
    ("C1_closed",    "Vocalised"): load("closed_vocab", "voiced"),
    ("C1_open",      "Silent"):    load("open_vocab", "silent", group="silent_parallel_data"),
    ("C1_open",      "Vocalised"): load("open_vocab", "voiced", group="voiced_parallel_data"),
    ("C1_non-par",   "Vocalised"): load("open_vocab", "voiced", group="nonparallel_data"),
}

for k, v in partitions.items():
    print(f"{k[0]:<12} {k[1]:<10} n = {len(v)}")


## Table 5 — corpus summary

`|V|` is the deterministic tokenisation of this repo. Note the closed subset recovers
**61** word types where the original authors report 67; the discrepancy is discussed in
Sec. 4.1.2 and traced to undocumented date/numeral expansion.

`transform_dataset` drops utterances whose transcript yields no tokens (bare
punctuation), so the analysable count is slightly below the raw count — 1,582 of 1,588
in the parallel open subset. Table 5 reports the **raw** recording counts.


In [ ]:
def vocab_size(df, token_based):
    tdf, vocab, _ = transform_dataset(
        df.copy(), token_based=token_based, load_emg_data=False
    )
    return len(vocab), len(tdf)


rows = []
for name, df in [
    ("C1_closed",     partitions[("C1_closed", "Vocalised")]),
    ("C1_open",       partitions[("C1_open", "Vocalised")]),
    ("C1_non-par",    partitions[("C1_non-par", "Vocalised")]),
]:
    v_word, n_analysable = vocab_size(df, token_based=False)
    v_tok, _ = vocab_size(df, token_based=True)
    rows.append({
        "Dataset": name,
        "Utterances / mode (N)": len(df),
        "Analysable (non-empty)": n_analysable,
        "Vocab |V| (word)": v_word,
        "Tokens": v_tok,
    })

table5 = pd.DataFrame(rows)
display(table5)


## Table 9 — utterance duration statistics

Duration is the EMG signal length divided by the 1000 Hz sampling rate. Loaded with
`mmap_mode="r"` so only the array header is touched — no 6.4 GB read.


In [ ]:
def durations_s(df):
    return np.array([
        np.load(p, mmap_mode="r").shape[0] / SAMPLE_RATE for p in df["emg_path"]
    ])


rows = []
for (subset, mode), df in partitions.items():
    if subset == "C1_non-par":
        continue   # excluded from the study — not parallel
    d = durations_s(df)
    q1, q3 = np.percentile(d, [25, 75])
    rows.append({
        "Subset": "Closed vocab" if subset == "C1_closed" else "Diverse vocab",
        "Mode": mode, "n": len(d),
        "Mean": round(float(d.mean()), 2),
        "Median": round(float(np.median(d)), 2),
        "SD": round(float(d.std(ddof=1)), 2),
        "IQR": f"{q1:.2f}-{q3:.2f}",
        "_total_h": d.sum() / 3600,
    })

table9 = pd.DataFrame(rows).sort_values(["Subset", "Mode"], ascending=[False, True])
display(table9.drop(columns="_total_h"))

silent_h = table9.loc[table9["Mode"] == "Silent", "_total_h"].sum()
diverse_silent_h = float(
    table9[(table9["Mode"] == "Silent") & (table9["Subset"] == "Diverse vocab")]["_total_h"].iloc[0]
)
print(f"\nTotal silent speech, both subsets : {silent_h:.2f} h")
print(f"Diverse-vocab silent only        : {diverse_silent_h:.2f} h")


## LaTeX export


In [ ]:
OUT = REPO_ROOT.parent / "results" / "tables"
OUT.mkdir(parents=True, exist_ok=True)

(OUT / "table5_corpus_summary.tex").write_text(
    table5.to_latex(index=False, escape=False,
                    caption="Summary of the full C1 corpus.", label="tab:corpus")
)
(OUT / "table9_utterance_durations.tex").write_text(
    table9.drop(columns="_total_h").to_latex(
        index=False, escape=False,
        caption="Utterance duration statistics for the two parallel subsets of C1.",
        label="tab:durations")
)
print("wrote:", OUT / "table5_corpus_summary.tex")
print("wrote:", OUT / "table9_utterance_durations.tex")
